# 04_MPM_Signal_Analysis: 会合直前シグナルの逆張り vs 順張り分析

このノートブックでは、会合直前（Days_to_MPM ≤ 5）において、モデルの予測方向に対する順張りと逆張りの有効性を検証します。

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys
import os

# src読み込み用パス設定
sys.path.append('..')
from src.processing import load_and_clean_data
from src.features import generate_features
from src.pooling import pool_boj_data
from src.modeling import walk_forward_validation, calculate_metrics

%matplotlib inline
plt.rcParams['figure.figsize'] = (12, 6)

## 1. データ準備と検証の実行

In [ ]:
excel_path = '../data/BOJ_data.xlsx'
meeting_path = '../data/BOJ_meeting_history.csv'

df_clean = load_and_clean_data(excel_path, meeting_path)
df_feat = generate_features(df_clean)
df_pooled = pool_boj_data(df_feat)

horizons = [3, 5]
results = {}
start_date = '2024-01-01'

for h in horizons:
    target_col = f'Target_{h}d'
    print(f"\n--- Running Validation for {target_col} ---")
    res = walk_forward_validation(df_pooled, target_col, start_date=start_date)
    
    # Days_to_MPM をマージ
    days_mpm = df_pooled[['Date', 'Meeting_Index', 'Days_to_MPM']].drop_duplicates()
    res = res.merge(days_mpm, on=['Date', 'Meeting_Index'], how='left')
    results[h] = res

## 2. 会合直前データの抽出と4象限分析

In [ ]:
def analyze_quadrant(df, horizon, m_idx):
    sub = df[(df['Days_to_MPM'] <= 5) & (df['Meeting_Index'] == m_idx)].copy()
    if sub.empty: return None
    
    sub['Pred_Sign'] = np.sign(sub['Pred'])
    sub['Actual_Sign'] = np.sign(sub['Actual'])
    
    # 4象限集計
    quadrants = []
    for p_sign in [1, -1]:
        for a_sign in [1, -1]:
            mask = (sub['Pred_Sign'] == p_sign) & (sub['Actual_Sign'] == a_sign)
            group = sub[mask]
            quadrants.append({
                'Pred': 'Up' if p_sign == 1 else 'Down',
                'Actual': 'Up' if a_sign == 1 else 'Down',
                'Count': len(group),
                'Mean_Return': group['Actual'].mean(),
                'Std_Return': group['Actual'].std(),
                'Median_Return': group['Actual'].median()
            })
    
    res_df = pd.DataFrame(quadrants)
    res_df['Horizon'] = f'{horizon}d'
    res_df['M_Index'] = m_idx
    return res_df

quadrant_results = []
for h in horizons:
    for idx in [1, 3, 5]:
        res = analyze_quadrant(results[h], h, idx)
        if res is not None: quadrant_results.append(res)

df_quadrant = pd.concat(quadrant_results)
display(df_quadrant)

## 3. 戦略別の期待リターン計算

In [ ]:
strategy_results = []
for h in horizons:
    res = results[h][results[h]['Days_to_MPM'] <= 5].copy()
    if res.empty: continue
    
    # 順張り: 予測方向にポジションを取る
    res['PnL_Follow'] = np.sign(res['Pred']) * res['Actual']
    # 逆張り: 予測と逆方向にポジションを取る
    res['PnL_Reverse'] = -np.sign(res['Pred']) * res['Actual']
    
    strategy_results.append({
        'Horizon': f'{h}d',
        'Expected_Return_Follow': res['PnL_Follow'].mean(),
        'Expected_Return_Reverse': res['PnL_Reverse'].mean(),
        'Win_Rate_Follow': (res['PnL_Follow'] > 0).mean()
    })

df_strategy = pd.DataFrame(strategy_results)
display(df_strategy)

## 4. 累積仮想PnLチャート

In [ ]:
for h in horizons:
    res = results[h][results[h]['Days_to_MPM'] <= 5].copy()
    if res.empty: continue
    
    # M1に限定してプロット
    res_m1 = res[res['Meeting_Index'] == 1].sort_values('Date')
    res_m1['Cum_PnL_Follow'] = (np.sign(res_m1['Pred']) * res_m1['Actual']).cumsum()
    res_m1['Cum_PnL_Reverse'] = (-np.sign(res_m1['Pred']) * res_m1['Actual']).cumsum()
    
    plt.figure(figsize=(12, 5))
    plt.plot(res_m1['Date'], res_m1['Cum_PnL_Follow'], label='Trend Follow')
    plt.plot(res_m1['Date'], res_m1['Cum_PnL_Reverse'], label='Mean Reversion')
    plt.title(f"Cumulative PnL (Horizon={h}d, M1, Days_to_MPM <= 5)")
    plt.legend()
    plt.grid(True)
    plt.show()

## 5. MPM回別の予測方向と実績値の散布図

In [ ]:
for h in horizons:
    res = results[h][results[h]['Days_to_MPM'] <= 5].copy()
    if res.empty: continue
    
    plt.figure(figsize=(10, 6))
    sns.scatterplot(data=res, x='Pred', y='Actual', hue='Meeting_Index', palette='viridis', alpha=0.6)
    plt.axhline(0, color='black', lw=1)
    plt.axvline(0, color='black', lw=1)
    plt.title(f"Prediction vs Actual (Horizon={h}d, Pre-Meeting)")
    plt.grid(True)
    plt.show()